### **1. Distribuzioni di probabilità**
Genera 1000 valori casuali da una distribuzione gamma con parametro di forma pari a 1.
Suggerimento: il parametro di forma viene passato come primo argomento quando si "congela" la distribuzione.

Traccia l’istogramma del campione e sovrapponi la PDF della distribuzione.

Stima il parametro di forma dal campione usando il metodo fit.

Extra:
Le distribuzioni hanno molti metodi utili. Esplorali usando il completamento automatico con il tasto TAB.

Traccia la funzione di distribuzione cumulativa (CDF).

Calcola la varianza.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

#1
sample = np.random.standard_gamma(1, size=1000)

#2
plt.figure(figsize=(12, 8))

plt.hist(sample, bins=50, density=True, alpha=0.7, color='skyblue', edgecolor='black', label='Istogramma campione')

x = np.linspace(0, np.max(sample), 1000)
pdf_values = stats.gamma.pdf(x, 1)
plt.plot(x, pdf_values, 'r-', linewidth=2, label='PDF teorica (Gamma, shape=1)')

plt.title('Distribuzione Gamma: Campione vs PDF Teorica', fontsize=14, fontweight='bold')
plt.xlabel('Valore', fontsize=12)
plt.ylabel('Densità', fontsize=12)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

#3
stima = stats.gamma.fit(sample, floc=0)
stimaShape = stima[0]
stimaScala = stima[2]
print(f"Parametro di forma stimato: {stimaShape:.4f}")
print(f"Parametro di scala stimato: {stimaScala:.4f}")

#4
plt.figure(figsize=(10, 6))
xCdf = np.linspace(0, 8, 1000)
cdfValues = stats.gamma.cdf(xCdf, 1)
plt.plot(xCdf, cdfValues, 'b-', linewidth=2)
plt.title('Funzione di Distribuzione Cumulativa (CDF)', fontsize=14)
plt.xlabel('Valore', fontsize=12)
plt.ylabel('Probabilità Cumulativa', fontsize=12)
plt.grid(True, alpha=0.3)
plt.show()

### **2. Fitta i dati**
Prova a fittare i dati sottostante con le migliori curve, calcola il MAE e l'RMSE

In [ ]:
import numpy as np
temp_max = np.array([17, 19, 21, 28, 33, 38, 37, 37, 31, 23, 19, 18])
temp_min = np.array([-62, -59, -56, -46, -32, -18, -9, -13, -25, -46, -52, -58])

import matplotlib.pyplot as plt

months = np.arange(12)
plt.plot(months, temp_max, "ro")
plt.plot(months, temp_min, "bo")
plt.xlabel("Month")
plt.ylabel("Min and max temperature")


In [ ]:
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import root_mean_squared_error 

def modello_annuale(x, amp, shift, offset):
    return amp * np.sin(2*np.pi*(x + shift)/12) + offset

params_max, _ = curve_fit(modello_annuale, months, temp_max,p0=[(temp_max.max()-temp_max.min())/2, 0, temp_max.mean()])
params_min, _ = curve_fit(modello_annuale, months, temp_min,p0=[(temp_min.max()-temp_min.min())/2, 0, temp_min.mean()])

#1
fit_max = modello_annuale(months, *params_max)
fit_min = modello_annuale(months, *params_min)

#2
mae_max = mean_absolute_error(temp_max, fit_max)
rmse_max = root_mean_squared_error(temp_max, fit_max)

mae_min = mean_absolute_error(temp_min, fit_min)
rmse_min = root_mean_squared_error(temp_min, fit_min)

print(f"Max:\n ")
print(f" MAE {mae_max:.2f}\nRMSE {rmse_max:.2f}")
print(f"Min:\n ")
print(f" MAE {mae_min:.2f}\nRMSE {rmse_min:.2f}")

#3
fig, ax = plt.subplots()
ax.plot(months, temp_max, 'ro', label='max (dati)')
ax.plot(months, fit_max, 'r-', label='max (modello)')
ax.plot(months, temp_min, 'bo', label='min (dati)')
ax.plot(months, fit_min, 'b-', label='min (modello)')
ax.set_xticks(months); ax.set_xlabel('Mese')
ax.set_ylabel('Temperatura [°C]'); ax.legend()
plt.show()

### **3. Modello di regressione lineare dei seguenti dati**

 Calcola un modello di regressione lineare delle colonne mpg e disp del famoso dataset mtcars.

Dove: 

mpg = Miles Per Gallon → miglia per gallone, cioè una misura del consumo di carburante. Più alto è il valore, più efficiente è l’auto.

disp = Displacement → cilindrata del motore, in pollici cubici (cubic inches). Rappresenta il volume totale dei cilindri del motore. Più è alto, maggiore è la potenza potenziale del motore (ma anche il consumo).

In [ ]:
import pandas as pd

# Load dataset from URL
df = pd.read_csv('https://gist.githubusercontent.com/seankross/a412dfbd88b3db70b74b/raw/5f23f993cd87c283ce766e7ac6b329ee7cc2e1d1/mtcars.csv')

# Display the dataframe
df


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt
import numpy as np

#1
displacement = df[['disp']].values              
consumi = df['mpg'].values            
regressor = LinearRegression()          
regressor.fit(displacement, consumi)                  

slope, bias = regressor.coef_[0], regressor.intercept_
print(f"mpg = {slope:.3f} * disp + {bias:.3f}")

predictions = regressor.predict(displacement)            
r_squared = r2_score(consumi, predictions)             
print(f"R^2: {r_squared:.3f}")

x_range = np.linspace(df.disp.min(), df.disp.max(), 200)
plt.scatter(df.disp, df.mpg, label='osservazioni')          
plt.plot(x_range,regressor.predict(x_range.reshape(-1, 1)),label='linea di regressione')
plt.xlabel('Displacement')
plt.ylabel('Consumi')
plt.legend(); plt.title('Regressione: mpg vs disp')
plt.show()